# 模块化迁移系统整合开发记录

日期：2026-11-16。范围：工程整合与本地校准，不是新一批模型实验或独立测试成绩。

依据开题报告研究内容 1.1—1.4，将现有实验能力接入 `migration_system`：真实 LangGraph 主图、SSM、few-shot 生成、静态／AST TED／DOM／Puppeteer／SSIM／像素／样式／代表性交互、规则与模型修复、候选回退、API／数据库／工作台及人工台账。LangSmith SDK 摘要追踪、LoRA 数据准备／训练／adapter 推理及本地视觉嵌入有实际接入代码；外部资源缺失时明确不执行。

实现路径、命令、边界与最终内测结果以仓库根目录 `migration_system/SYSTEM_GUIDE.md`、`SETUP_PENDING.md`、`DEVELOPMENT_20261116.md` 为准。保留旧 Notebook、快照、Vue 源码及旧夹具，不覆盖历史结论。

## 新协议与不能混淆的结果

新协议 `modular-migration-v1` 最多一次生成、三轮修复。次数、每次输出上限和已知 usage 停止线不是硬计费上限。旧 G3 上界批准与旧请求保持原状，网络默认关闭；启用后创建任务仍需明确确认调用政策。

源基准警告不抑制；目标异常与设施故障分流；失败候选不可覆盖接受版本；未知模型调用不重发。人工结果绑定产物哈希、另行追加，不覆盖自动分数。

结构、视觉、交互与编译运行分别报告。UI = 100 × (0.5×SSIM + 0.3×(1−不同像素率) + 0.2×样式一致率)。结构与 UI 阈值 >95%，不同像素率 <5%。未配置交互标为未测，不代表所有行为等价；新口径不能与旧实验直接合并。

本轮没有云模型调用、LangSmith 上传、模型下载、LoRA 训练或历史 30 项复跑。默认 few-shot 是两个自建开发演示，不是测试集答案，不据此宣称准确率提升。

## 本地真实证据与部署状态

真实 Puppeteer 校准包含：同图、已知颜色差异、目标运行异常、SSM＋旧静态校验＋按钮交互，以及实际 Vue 工作台→Flask→临时 SQLite 提交。图控制／恢复／费用未知／回退等用明确标记的测试客户端，不将其结果冒称模型产物。

用户实际业务库只读核对为 `0006_generation_approvals`，三条源输入；原 SSM 的 request／receipt／ssm 三份 SHA-256 未变。新代码新增 0007 四表，未自动迁移实际库。统一部署需显式 init-db 后启动 API 与 migration-worker。

后续真实模型、LangSmith 与 LoRA 的外部验收应另追加记录，不回填为本轮已完成。

## 内部验证结果

全套当次 330 项通过（595.297 秒）；当前模块收口专项 32 项通过（59.724 秒），训练数据专项 3 项通过。当前收集为 333 项，新增用例包含在专项中，没有声称用同一命令一次跑过全部 333 项。专项与全套重叠，不能相加当样本数。

原 P1 40 项、前端 Node 22 项、像素 Node 1 项通过；TypeScript 与 Vite 构建通过。真实浏览器校准与真实前端临时库提交通过。源码／锁文件／日志哈希摘要见 `migration_system/verification_20261116.json`。

收口边界包括：被拒绝规则转模型修复、规则轮次不占模型调用名额、超限响应保留用量且不能借收据重放放行、嵌套 template 不截断 SFC。上述是工程控制与固定样例证据，不是模型迁移成功率。

## 首次真实提交阻断与 CommonJS 源加载修复

用户启用模型网络后提交 DeliveryPhaseCard.vue。任务 `3a5205caedf0413493256822a1928be9` 在 preflight 阻断：`unsupported_source_sfc`，模型开始 0 次、未知调用 0 次、已知 Tokens 0。上传源码与仓库原文件 SHA-256 均为 `2a057ec6591b35d3044b9b77106e849ccea48e0ac407ab39d34d7e37fceda16b`，不是源码损坏。

根因是新浏览器适配器仅支持 export default，未兼容原样本的 module.exports Options 对象。修复位于 browser/evaluate.mjs 的源加载适配层，不涉及生成／修复 Prompt。仅支持单一顶层 module.exports = { ... }，浏览器内提供模块作用域；混合／重复导出、工厂调用和外部 import 仍阻断。未修改原 Vue、上传文件或旧任务证据，也未降低阈值。

新增两种导出预检正例、歧义／依赖阻断反例、真实浏览器 CommonJS 点击更新测试。测试夹具强制关闭模型网络，不继承用户启用状态。重启 API／migration-worker 后需新建真实任务；旧阻断任务不能改绑成修复后通过。验证结果另见 migration_system/DEVELOPMENT_20261116.md。

## FocusCard 评估误判修正与新旧离线对照

原真实任务 f3cf07b0c7e5462aaf1b1cd5a781b37a 为 failed / repair_budget_exhausted，模型开始 3 次、已知 Tokens 41391。最终接受的是第 1 轮产物，SHA-256 为 0ced63ec865d38bedf3cdc4d5a102780d35d48684639af75900675f69decaced。原报告静态 class fail、SSM/San 结构 57.14%，实际 DOM／视觉均 100%。

修正共享旧 class 检查器对条件字符串空白的误判，保持 token 边界和完整条件验证；修正新 AST 投影对 San 三元插值、固定 class、相邻文本分段的处理，投影版本为 v2；样式路径按元素兄弟编号，避免缩进空白导致错位。阈值、TED 代价、像素与 UI 公式不变，未修改生成／修复 Prompt，也未手改生成代码。结构差异附具体路径／标签；依赖指纹覆盖实际 Node 检查器。

用原字节产物离线重评：旧验证器直接调用与新适配层均 pass；SSM/San 与源/San AST TED 均为 0，DOM 100%，UI 100%；旧 VisualEvalStage 的 DOM／文本相似度也均为 100%（不冒充像素评估）。另行补充开始／停止两次点击，代表交互 pass。历史空场景仍 not_run，不回填。

本轮新增模型调用 0。原失败任务及其用量不变，离线重评另存为 data/experiments/evaluator_calibration/2026-11-16_modular_focus_card_evaluator_replay_v1；补齐多 token 源 class 键的空白边界后，最终实现另存为同级的 2026-11-16_modular_focus_card_evaluator_replay_v2，不覆盖 v1。包含历史索引核验、原报告／源码／产物副本、新截图、comparison.json 和 SHA256SUMS.json。它是开发校准，不是独立测试或新模型成功样本；测试执行结果见 migration_system/DEVELOPMENT_20261116.md。

## ChoiceChips：实际 DOM 诊断与空 class 受限比较

原任务 6d2eb1d3cc034c88b1b4167728334c8c 为 failed / repair_budget_exhausted，模型开始 3 次、已知 Tokens 36725。接受产物是第 2 轮，SHA-256 为 6b4bd4b84d866d93e37a0fdbb742ea3d6ea2ce52d39a22088a1c4a43121bce29；原报告 SHA-256 bb45e4d3745e03fdfaf8acc5d4e31a7b79c53fc6bcc07bc6055abe717322994a。SSM/San AST 已为 100%，原始 DOM 83.33%，差异为两个未选中按钮的空 class／缺失 class。原反馈未给实际 DOM 差异，导致无明确定位的修复消耗。

补齐 DOM／动作失败的路径和属性诊断，并在候选保护不退化的前提下识别 DOM 严格改善。新增 empty_class_presence_v1 受限比较策略：双端保守 AST／模板／CSS 检查获准后，仅在比较副本里归一化精确空 class；存在属性选择器、DOM 观察、反射、未知依赖或分析失败则保留严格比较。原始树、原始 TED、策略原因和改变路径全部保留，不证明任意外部 DOM 观察者等价。

原产物离线重评：静态、编译运行、源基准、SSM/San 结构、视觉通过；原始 DOM 83.33% 不变，有效比较 100%，UI 100%。另补充即时、每日、关闭、恢复每日四次选择，均通过。辅助源模板 AST 87.5% 保留，不回填历史空场景 not_run。没有修改生成代码、Prompt 模板、阈值或预算，实际修复消息会包含新增诊断。

本轮新增模型调用 0。原失败任务和 36725 Tokens 账本不变。46 条历史哈希引用（含重复引用）核验后，独立证据存于 data/experiments/evaluator_calibration/2026-11-16_modular_choice_chips_dom_replay_v1，包含原报告／源码／接受产物副本、截图、comparison.json、SHA256SUMS.json 和 REPORT.md。属于开发校准，不是新模型成功样本或独立测试；测试结果见 migration_system/DEVELOPMENT_20261116.md。

## ExpenseTracker：语法优先闭环与有边界的通用修复

原任务 5c2fc8916fcd4b8bb65f32ea28e20488：blocked / model_usage_stop_reached，模型开始 3 次，已知 Tokens 175512。Vue 源哈希 47995c773b5698e5f4fda5b5e854b27648c3b1e030a845ab19790930ad564736。不是已证明 Vue 源码错误：生成的 San 使用无效 <s-for=…> 包装，前两轮脚本方法之间缺逗号。适配层只返回 invalid_target_sfc，语义分析级联，未优先修好语法。

新增 target_syntax_first_v1：SFC／Babel／San 详细诊断，行列／片段及位置精度；语法失败时推迟语义、双端浏览器、结构与视觉检查。新增两条无组件名白名单的局部规则：源码唯一匹配循环／key／元素后把伪包装改为单子元素的 s-for 属性；仅在 Babel 指向缺逗号且完整新 AST 确认相邻对象方法边界时补一个逗号。字符串／注释／插值、未知来源、多子元素、嵌套包装及丢失元素不猜测修复。

旧生成 Prompt、旧语义修复 Prompt 入口保留；新增聚焦语法消息分支，只传当前目标、原 Vue 与具体解析错误，不重复完整 SSM。规则候选必须复测；语法首次通过后新增的语义观测不会误当成回退，仍须所有必需维度通过才能判成功。已通过维度／源基准保护不取消，预算和质量阈值未放宽。

离线重放先复制并核对 24 个历史任务文件及源输入（共 25 文件），无模型调用、不更新任务。最终证据 data/experiments/evaluator_calibration/2026-11-16_modular_expense_tracker_syntax_replay_v4。初始版诊断 10→3，因缺 option 子元素继续阻断；第 1／2 轮产物诊断分别 10→0、9→0，真实编译运行通过，但静态语义／DOM 仍失败；代表交互仍 not_run。不得把语法改善写成整个样本迁移通过。

v1 保留沙箱内浏览器超时的未完成现场，v2／v3 为中间完整证据，v4 对应最终位置精度标注。原失败和 175512 Tokens 不抹除，未改原 San、源 Vue、用户配置或手工测试页。最终模块回归 70 项通过、166.829 秒，含新增语法专项 16 项；工程用例不当作迁移成功率分母。30 个证据 payload 与实现指纹核验通过。聚焦语法消息尚未发送真实模型，不宣称模型修复成功率已验证。规则、测试及部署细节见 migration_system/SYNTAX_REPAIR.md、DEVELOPMENT_20261116.md。这是开发回归，不是独立测试或新模型成功样本。

## ExpenseTracker：语法之后的共享规则／验证闭环

继续使用原失败任务 5c2fc8916fcd4b8bb65f32ea28e20488 的接受 revision 1，经上一轮语法规则得到的固定输入 6b513daf6bb3e8a12738516f01d2c8b34bd0be6f675171dee84f1fea2065438a，不重新生成或手改原产物。Enter 事件复用旧规则；补齐类名字符串分隔、数值样式对象／CSS 插值的来源证明及公开签名恢复、已有 watcher 时 mounted 调用移动到其后的阶段规则。修改位于 migration_pipeline/utils 的共享实现，新系统通过 residual_v2 适配入口复用，而非组件名特判。

规则有明确边界：只证明匹配引用的类名分隔符，不证明任意动态类型转换；合法字符串拼接不误报。样式只支持单尺寸属性、显式匹配单位、可证明数值只读叶方法，保留源公开调用方式并隔离模板依赖；未知调用、脚本／多模板引用、单位或方法体变化不自动修改。watcher 字段、选项和无参回调必须匹配源，无其他早期／未知启动行为，不复制加载调用。

离线候选 SHA-256 9b8be8e3a0adcd7bd1f0b03246601b21441954a281b281ad31acea542fd1e34f：当前静态错误 7→0，编译运行／源基准／静态／结构／视觉通过，原始及有效 DOM、SSM/San 结构均 100%，UI 100%、SSIM 1、不同像素 0。辅助源 Vue／目标模板 AST 保留 96.10%（三个 binding:key／trackby 表示位置差异），未改为 100%。门控 strict_improvement 接受，规则重复运行不再改动字节。另加非 Enter 不添加、Enter 添加、收支切换、过滤／清除、主题及删除共 15 步诊断交互全部通过；旧任务的交互 not_run 不回填。

证据 data/experiments/evaluator_calibration/2026-11-16_modular_expense_tracker_semantic_replay_v3：先核对并冻结上轮 30 个 payload，保留实现、候选、双端截图、逐步原始快照、差异与门控。v1／v2 为中间证据，不覆盖。原失败状态和 175512 Tokens 账本保留，本轮模型调用 0、无 LangSmith 上传、未改用户配置／原 Vue／原 San／手工测试页，未修改 Prompt 或增加预算。

最终分别验证新增规则 16 项通过（41.386 秒）、模块 70 项通过（320.199 秒）、旧 TaskManager 冻结闭环独立专项 1 项通过（186.200 秒）。开发中未全通过的中间合并运行不算通过证据，夹具修正和复测说明见 DEVELOPMENT_20261116.md。不把工程测试数量当迁移样本数，也不把冻结开发产物的离线候选通过计为新模型或独立测试成功。真实模型端到端、任意输入／全部公开调用和跨刷新持久化不在本次证明范围；部署需重启 API／Worker 后新建任务。

In [ ]:
# 只读检查；不初始化数据库、不启动 Worker、不调用模型、不打印密钥。
from pathlib import Path
import subprocess
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'migration_system').is_dir())
python_exe = root / 'migration_system/.venv/Scripts/python.exe'
result = subprocess.run([str(python_exe), 'migration_system/scripts/run_system.py', 'system-status'], cwd=root, capture_output=True, text=True, timeout=30)
print(result.stdout or result.stderr)

## 2026-11-17：从本次初始生成出发的完整规则闭环

原任务 b2a59e959191487fa4c3acbe945f04cb（ExpenseTracker）仍为 failed / repair_budget_exhausted，真实模型调用 3 次、已知 Tokens 80802，不改历史结果。初始语法诊断 10 条，第 1 轮旧规则降为 4 条；后两次模型请求 messages 相同，候选均无进展而被拒绝。这不是视觉分数或源 Vue 错误。先冻结 29 个任务文件与源输入，共 30 文件；源码 SHA-256 47995c773b5698e5f4fda5b5e854b27648c3b1e030a845ab19790930ad564736。

本轮扩展来源可证明的缺失 option、多处方法逗号／闭合标点恢复；保留已接受父版本，附带同父版本拒绝候选的哈希、原因和实际诊断；重复完整修复 messages 在模型调用前阻断。生成 Prompt 和旧语义 Prompt 模板未改，语法专用消息新增拒绝反馈，预算／Token 停止线／阈值不变。规则不看组件名，不手改生成文件。

完整重放进一步暴露字符串 CSS 包络、非 San watcher 配置和 missing_watch 诊断未调度对应规则的遗漏；已同时补共享 CJS、RepairStage._deterministic_repair_plan 的触发映射及正反例。watcher 转换只支持源明确 deep:true 的精确零参回调和来源匹配启动序列，浅 watcher、immediate、未知副作用继续阻断。

语法最终证据 data/experiments/evaluator_calibration/2026-11-17_modular_expense_tracker_syntax_closure_v3：4 个冻结 revision 均通过语法，36 payload。使用其初始 revision 0 的规则输出继续语义重放，最终证据 data/experiments/evaluator_calibration/2026-11-17_modular_expense_tracker_semantic_closure_v3：诊断总数 9→0（包含结构诊断，不全是静态错误），静态／编译运行／源基准／结构／视觉通过，SSM／San、原始与有效 DOM、UI 均 100%，SSIM 1、不同像素 0；另加固定 15 步动作全部通过，不回填原空动作。辅助源／目标 AST 仍 96.10%，未修改指标。候选门控接受、规则幂等，最终候选 SHA-256 0c0d2cde569db4f1fce50b680da732a2b7888e06ce90b19109bfa660e9e00ff3；新证据封存 62 payload。

最终完整模块回归 79 项通过（490.876 秒），含初始历史响应／SSM注入临时数据库后，真实 LangGraph→规则→候选门控→浏览器 15 动作的路径；额度仍为 3 轮，修复 revision 全来自规则，没有额外模型修复。生成调用使用明确测试响应，不用测试替代的评估分数。共享语义／实际调度专项 20 项通过（75.513 秒）；原流水线回归 78 项通过（383.318 秒）；语法／反馈专项 24 项通过（53.290 秒，已包含在模块回归）。这些是有重叠工程测试，不相加为迁移样本分母。

保留 syntax v1／v2 和 semantic v1／v2 中间结果，后两者静态仍剩 3／2 条，不计成功；受限环境浏览器启动失败也不计质量通过。原任务与源文件哈希未变，本轮新增真实模型调用 0、LangSmith 上传 0，不修改真实数据库、配置或手工页。这是开发失败的离线闭环，不是新模型迁移成功或独立测试证据；未证明跨刷新持久化、任意 props／全部公开调用等价。部署需重启 API 与 migration-worker，然后新建任务；不可恢复旧任务改写历史。细节见 migration_system/SYNTAX_CLOSURE_20261117.md 和 DEVELOPMENT_20261116.md。

## 2026-11-17：源模板 HTML 空元素解析入口修复

原任务 3d3d69c7760045e1bd0c741f7cf3638b（InvoiceLineEditor.vue）为 blocked / browser_evaluation_blocked，模型调用 0、Tokens 0。源源码 SHA-256 9b50005a20314dd9c1f52db9496c6092d8f2838958955be0fa0350db7c77a8df。新评估器的 Vue 模板 AST 入口使用 compiler-dom.baseParse，缺少 HTML 平台空元素规则，误将合法 input 识别为缺少结束标签。SFC 解析、HTML 模板解析及编译的独立对照均通过，不将其归因为源代码或模型问题。

源投影改为 compiler-dom.parse；San 目标解析仍保持原入口，不加忽略错误的回调或正则补标签。input／img／br／hr／wbr 的普通和自闭合写法均覆盖；div／span／textarea／strong 等真实缺失或错配结束标签继续生成前阻断。新增源模板／SFC／脚本错误代码，报告携带绝对 UTF-16 位置、1-based 行列、定位精度和有限源码片段；Python 按已知源码重算，过滤外来字段。工作台转义显示片段，模型未调用时明确提示，LangSmith 不上传源码诊断。

先冻结原 request、failure 与源输入，共三个文件；修复前证据 data/experiments/evaluator_calibration/2026-11-17_modular_source_void_elements_preflight_v1，修复后为同级 v2。源码字节不变，源 preflight 从 blocked 变为 completed；v2 10 个 payload、当前实现指纹及三个原件哈希均核验。重放不启动渲染浏览器、不调用模型、不修改真实数据库或历史任务；没有生成该组件的 San，源预检查通过不能记为迁移质量通过。

专项 12 项通过（33.895 秒），含真实解析器、临时数据库／LangGraph 零调用阻断、实际工作台诊断显示及双端浏览器；完整模块回归 91 项通过（241.352 秒），包含专项，不相加为迁移样本数。前端 TypeScript／Vite 构建和 Node 22 项测试通过。本轮未重跑根目录旧流水线全套。开发中静态 input value 的框架属性表示差异曾使夹具失败，改用 placeholder 隔离空元素解析后重新验证，未放宽 DOM 比较规则；该独立属性差异不宣称已修复。

未修改 SSM、生成／修复 Prompt、预算、阈值、用户配置或手工页，未删除适配器。原任务和费用原样保留；新增真实模型调用 0、LangSmith 上传 0。这是工程回归与开发校准，不是新的模型迁移成功样本或独立测试。记录见 migration_system/SOURCE_TEMPLATE_PARSING.md 和 DEVELOPMENT_20261116.md；部署重启 API 与 migration-worker、刷新页面后新建任务，无需 init-db，后续真实迁移仍需单独验证。

## 2026-11-17：ProductFilterPanel 任意公开键能力边界与控件实际值诊断

原任务 ea3808cfe9ff4ae7b301e5c8d593e3aa：failed / repair_budget_exhausted，模型 3 次（生成一次、模型修复两次；另有一轮规则修复），Tokens 87999。共享规则补齐 props 默认值及循环，模型恢复混合节点顺序，最后一次响应与接受 revision 2 完全相同而被拒绝。源语法、编译运行和最终视觉通过；静态剩 source_dynamic_assignment_unproven，原始 DOM 8 处 value／空 class 属性差异，交互原配置为空，not_run 不改写。源 SHA-256 aed0d49a2bab60b3103e50beb0236320d35f7fb9d8bd2341db36a65be65e817a。

保留用户既定的任意公开键契约。this[key]=value 与 this.data.set(key,value) 对字面点号键、Symbol、实例成员命名空间不能直接证明等价；原生 San 反例确认点号键为路径而非字面实例属性。没有把 setter 降级为六个固定键，也没有开发完整动态实例兼容层，故本轮不宣称语义能力已修好。共享 method_interface_semantics 导出原识别器，新源能力检查复用它，在语法检查后、SSM／模型调用前以 source_dynamic_key_contract_unsupported 明确阻断，源代码合法与解析错误分开提示。未知方法形状仍交由原验证器，不把未匹配视为完整支持证明。

初始态新增 live value／checked 检查，多选 select 采集全部选中值；状态不同即产生 rendered_control_state_mismatch 并使结构失败。初始与动作后的实际值差异写入报告，原 DOM 不改写，值相同不能豁免 value 属性／class 存在性。空 class 分析中的另一处 baseParse 改为 HTML parse，但动态成员／DOM 反射／未知调用继续拒绝归一化，ProductFilterPanel 未获批准。工作台新增原始诊断和初始控件状态显示，不改阈值或 Prompt 模板。

先冻结 45 个原任务文件、源输入和 freeze.json，47 payload，目录 data/experiments/evaluator_calibration/2026-11-17_modular_product_filter_failure_frozen_v1。最终独立证据为同级 2026-11-17_modular_product_filter_contract_replay_v4，21 payload，使用原接受 revision 2、未手改生成文件。11 步限定动作的控件实际值全部一致，双端含初始态 24 条明确商品列表预期全部符合；但严格 DOM 继续失败，新增场景的整体 interaction 为 fail，static 仍 fail，passed=false。不能把限定 UI 观察等同于任意公开方法或独立迁移成功。

v1 为受限环境浏览器失败的未完成现场，v2 因场景预期断言失败退出；v3 保存错误预期下的完整观察。初始预期误用输入数组顺序，并假设 clear 重置 sort；v4 按源实际名称排序及 clear 保留已选排序修正夹具后重测。生产代码／源代码没有为夹具而修改。

最终新增专项 11 项通过（33.120 秒），完整模块 102 项通过（671.729 秒，已含专项），原共享方法接口 16 项通过（463.020 秒）；前端构建和 Node 22 项通过，不相加为迁移样本数。初次专项模块路径错误、受限环境浏览器失败及其修正／最终完整复测独立记录，根目录其他测试未全量重跑。47 个冻结 payload、21 个最终重放 payload、当前实现和 46 个原件哈希核验。部署细节见 migration_system/DEVELOPMENT_20261116.md 与 PUBLIC_KEY_CONTRACT_20261117.md。原失败、账本和 not_run 保留；本轮真实模型调用 0、LangSmith 上传 0，无真实数据库写入、commit 或 push，不计论文迁移通过率。该组件仍需完整语义能力实现或另获批准的新契约，不能仅靠重启或增预算复跑放行。

## 2026-11-17：独立目录重构与实验能力内聚

按用户要求，不再把跨根目录只读适配当作独立系统。开题中的四维 SSM、few-shot／LoRA、LangGraph／LangSmith、结构 TED、Puppeteer／SSIM／像素／样式及规则＋模型候选闭环保留。内置 engine/MANIFEST.json 记录 98 项来源：96 项逐字节保留，另两项为惰性包入口和纯生成 builder／常量提取，不导入旧 Flask 或旧完整编排器。旧 SSM、migration_pipeline、local_server、数据集及冻结结果没有修改。

生成／修复职责从任务账本服务拆分到 stages/generate.py 和 stages/repair.py，图仍唯一负责条件分派、复测及候选接受。真实迁移 UI 默认不挂载或查询开发夹具，旧页按需延迟加载。worker 指向真实迁移，旧夹具显式用 development-worker；历史 README／指南归档，实验重放脚本移回 tools/migration_system_history，独立系统运行不再需要它们。

设置的 engine_root 在本目录，repository_root 仅是兼容字段名，不会回退父目录。新默认数据库／检查点在 var/runtime，源、SSM、San、收据／报告／截图在 var/artifacts。用户 system.toml、旧 var/business.sqlite3 及父目录旧产物原样保留；没有自动迁移、初始化实际库或覆盖旧实现绑定。新路径为空不是旧数据被删除，部署需显式按当前 SYSTEM_GUIDE 操作。

5 项来源等价对照通过（96.340 秒）：SSM 完全相同，生成 messages 完全相同，静态诊断／规则／修复 Prompt 相同，拆分修复函数 AST body 一致，源文件逐字节归属可核验。生成边界 18 项通过（17.972 秒）；前端 TypeScript／Vite 构建和 Node 22 项通过。完整系统 408 项执行 2107.899 秒，406 通过、2 个旧夹具 CLI 名称用例失败；修正为 development-worker 后完整任务链路 22 项通过（67.098 秒）、配置／入口 29 项通过（15.749 秒），新增真实 Worker 别名断言。收尾将普通测试也改为内部 engine 后，SSM／登记图／配置共 76 项通过（247.747 秒），完整模块化专项 102 项通过（524.307 秒，无跳过）。当前历史语料存在，旧失败闭环用例没有因独立化被跳过；仅在独立分发缺少旧语料时明确标记跳过。只读 check、98 项清单、独立副本运行时指纹及封存证据哈希核验通过。没有声称同一命令一次跑过新增后的全部 409 项，各组重叠不加成迁移样本数。

独立性验证真实复制源码、配置、依赖和前端构建到仓库外临时目录；禁止读取旧 checkout 源码，所有载入系统模块须在副本内。临时 API、数据库、SSM、真实 LangGraph、实际规则／校验／浏览器、修复候选接受和下载通过；使用明确的模型响应测试替代，两次 fixture exchange、真实云调用 0，六个配置维度通过。另核对合法空元素、真正语法错误及已知不支持契约在模型前阻断。证据 migration_system/docs/verification/2026-11-17_standalone_refactor/standalone.json；不是新模型成功样本，不证明总体通过率提升。

保留前几次工程验证发现的说明：Windows 换行导致纯 builder 指纹口径不匹配，已按原约定 LF 归一化修正；测试不能要求固定模型次数，因为原规则可能先修好，也不能把阈值内小色差当作必须修复。最终用显著布局／背景缺陷验证实际修复分支，没有降低阈值或写假分数。没有训练、LangSmith 上传、付费调用或真实队列／数据库写入；既有未证明语义及严格 DOM 问题不因重构被豁免。详见新 README、docs/ARCHITECTURE.md、docs/REUSE_MATRIX.md 和 DEVELOPMENT_20261116.md。

## 2026-11-17：已授权的内部公开实例代理兼容性原型，未接入生产

用户明确允许内部公开实例代理，先验证兼容性再接入。研究实现位于 migration_system/research/public_instance_v1，不由主 Worker 加载，不改变生成／修复 Prompt、SSM、原引擎、规则、默认组件导出或生产契约。公开业务 receiver 与原生 San receiver 分开，状态指针由闭包／加载方句柄持有；任意字符串、Symbol 和需 ToPropertyKey 的键不受固定业务键白名单限制。当前元数据／方法来自合成夹具，尚非自动 SSM 编译或真实组件生成。

原生 Vue 3.5.45 与 San 3.15.6 的 73 个有限场景一致，覆盖字面点号／括号键、Symbol、转换顺序与异常、额外上下文值、同名 data/watch 等属性、只读写入和捕获方法覆盖。两张隔离 Edge 页在公开 data=null、watch=42 后继续真实点击，文本同为 0→1→11，状态指针保持、dispose 成功；San 页无 Vue 运行时、双方无 warning/error。不是完整视觉指标或真实组件验收。

4 个已观察差异仍保留：声明状态对象响应式身份、内部实例图读取、setValue('$',value) 后的公开读取、公开 $watch API。深层别名／ref、完整 computed 缓存／反射／调度、组件树及自动生成／加载等未充分验证。arbitrary_public_key_contract_complete=false，production_admission_eligible=false；现有 source_dynamic_key_contract_unsupported 不解除，不缩小键或值的契约来计成功。

最终原型专项 5 项通过（14.445 秒），既有公开键／控件专项 11 项通过（24.430 秒）。前两次 5 项（12.163／10.224 秒）是中间验证，不叠加。初始 warning 前缀比较问题修正，真实 __proto__ 差异按 Vue 默认 globalProperties 继承键 defineProperty 分支修复原型并新增反例。6 个源码／结果 payload 以 SHA256SUMS.json 封存于 migration_system/docs/verification/2026-11-17_public_instance_proxy_v1；原生产运行时指纹保持。

无云模型、训练、LangSmith、实际库写入、commit 或 push，不修改旧样本／费用，不算论文迁移成功率。后续先关闭反例并验证生命周期连接，再形成可审计生成／规则／校验接入版本。当前不需重启或付费复跑。详见 migration_system/docs/PUBLIC_INSTANCE_PROXY_V1.md 与 DEVELOPMENT_20261116.md。

## 2026-11-17：公开实例代理 v2 研究修复与边界验证（未接入生产）

沿用用户‘允许内部公开实例代理，先验证兼容性再接入’的授权。本轮不改 v1 源码及六项 payload 封存，不改生产生成／修复 Prompt、引擎、配置、历史产物或费用。实现位于 migration_system/research/public_instance_v2，详细说明见 migration_system/docs/PUBLIC_INSTANCE_PROXY_V2.md。

v2 新使用已安装的独立 @vue/reactivity 3.5.45 内核，这是新增候选依赖，不能称为完全无 Vue 依赖；目标页没有 Vue 组件／渲染运行时。保留源 raw data、共享／循环对象及 ref 身份，补充 computed 缓存、$watch 和逻辑实例关系。通过本实例 San Data 的实际读取订阅刷新嵌套别名／逃逸 ref 的 DOM，不额外遍历或重复求值 getter，不修改全局 San 原型；dispose 恢复自有描述符。

最终原生对照有 94 个有限场景一致。实际 Edge 四页为 source／target × 普通／ref 根：真实点击后计数 0→1→11；逃逸 ref 更新后为 21，嵌套别名更新后为 9。普通场景每页 6 条、ref 场景每页 8 条真实 pre/post 回调，值及回调时 DOM 文本一致，不以两个空列表当作时序通过。8 项边界控制通过，包括单次通知、raw 身份、借用接收者、描述符恢复、不兼容内核构建提前拒绝等。

最终 v1／v2 合并工程回归 13 项通过（32.993 秒，包含 v1 原有 5 项及 v2 新增 8 项）；既有公开键／控件回归 11 项通过（28.267 秒）。此前 13 项 19.760 秒是中间验证，不叠加。未全量重跑系统或 30 组件，以上不是论文迁移成功样本数。

仍观察到 private-instance-renderer-vnode-graph 差异：逻辑实例记录不等于完整 Vue 私有渲染记录，vm.$.vnode 未实现。原生 San 嵌套不可变写入、silent／force、子作用域、完整生命周期及异常／调度策略仍未闭合；不支持的路径明确拒绝。尚未从 SSM 自动编译真实组件、导出正式 SFC 或接入真实加载器。保持 arbitrary_public_key_contract_complete=false、production_admission_eligible=false 及 source_dynamic_key_contract_unsupported，不缩小任意公开键契约来放行。

源码、测试及 native／browser／controls 最终结果共 8 payload，以 SHA256SUMS.json 封存在 migration_system/docs/verification/2026-11-17_public_instance_proxy_v2。中间结果没有逐轮绑定源码，不能充当最终版本证据。开发中实际发现并修复 cleanup 接收者、错误处理返回值污染 getter、状态变而 DOM 不变、过早读取 watcher 日志及重复根通知问题，未抹去这些失败。

本轮无真实模型、LangSmith、训练、实际数据库／队列写入、commit 或 push。现在无需重启或付费复跑；应先关闭剩余边界，再验证可审计的真实组件编译／加载连接。研究兼容性结果不等于生产任意公开键契约已实现。


## 2026-11-17：公开实例代理 v3 原生嵌套控件写入（研究，未接入）

继续用户允许内部公开实例代理、先验证兼容性再接入的授权。v2 的业务方法嵌套赋值成功不代表 San 原生双向控件 Data.set 路径成功；v3 在 research/public_instance_v3 补充后者，保留 v1/v2 源码与封存，生产 Prompt、引擎、规则、阈值、配置及旧产物不动。仍显式使用独立 VueReactivity 内核，目标没有 Vue 组件／渲染运行时，不能称为完全无 Vue 依赖。

控件写入经源响应式状态执行，避免原生 immutableSet 复制路径破坏共享别名、数组项和 ref 身份。一次控件写入按受到影响的根收集合并通知；同步 watcher 侧写其他根或同根字段不丢失，保留调用方 option 身份。已测 silent／silence／quiet 只抑制 San 通知，不关闭源 watcher；相同值不通知，force 可显式通知。监听器回写放在内部写保护之外，避免状态分叉。采用根级失效事件，不声称原生 San 嵌套 change 事件逐字节等价。

v3 实现上原有 94 个有限原生对照一致；新增 5 项嵌套写入 Vue／San 对照，覆盖别名、动态数组下标、ref、字面点号／空字符串／Symbol、同步 watcher 侧写；8 项原生边界控制成立。两张隔离实际 Edge 页通过输入框、复选框反复开关、下拉框和字面点号键输入，公开 data=null／watch=42 后控件仍可用，最终 dispose=null 不破坏清理；每页 7 条实际同步 watcher 回调及 DOM／状态逐条一致，无 warning/error。不是两个空列表的比较，也不是完整像素视觉测评。

最终 v1/v2/v3 合并工程回归 17 项通过（21.334 秒，含旧 13 项及新增 4 项）；既有生产公开键／控件 11 项通过（24.760 秒）。此前专项 4 项 5.604 秒是中间验证，不叠加。未全量重跑系统、30 组件或真实模型，工程测试不计论文迁移成功样本。

保留真实失败说明：首次普通权限浏览器等待超时；随后隔离浏览器中发现源 Vue 页三击未全选，只删除末字符并形成 staredited。补充失败时状态和 DOM 诊断，改明确 Ctrl+A／Backspace／逐字输入后两页通过，没有改源组件或迁移判据。只有最终包绑定最终源码，中间结果不冒充最终证明。

最终 native／native_writes／browser 三份结果及五份源码／测试共 8 payload，以 SHA256SUMS.json 封存至 migration_system/docs/verification/2026-11-17_public_instance_proxy_v3；详见 migration_system/docs/PUBLIC_INSTANCE_PROXY_V3.md。

仍未完成私有 vnode／渲染实例图、完整生命周期／异常／调度及子作用域、真实 SSM 自动编译／SFC 导出／加载器接入；development 内核触发钩子限制未解除。缺失父对象和同步 watcher 中原生 setter 重入仍明确拒绝，通知收集不保证原子回滚。保持 arbitrary_public_key_contract_complete=false、production_admission_eligible=false 和生产 source_dynamic_key_contract_unsupported，不缩小任意公开键契约。无模型、LangSmith、训练、实际数据库／队列写入、commit 或 push，当前无需重启或付费复跑。


## 2026-11-17：公开实例代理 v4 渲染／nextTick 调度（研究，未接入）

继续用户先兼容验证再接入的授权。以同一真实浏览器探针在旧 v3 上复现后修复，旧代码／封存及生产引擎、Prompt、规则、阈值、配置、源 Vue 和产物保持；实现位于 migration_system/research/public_instance_v4。

8 个有限场景中，旧 v3 仅 3 项一致、5 项不同：没有 watcher 或只有 pre watcher 时 tick 早于 DOM，回调读取旧 DOM，post watcher 侧写导致顺序倒置，同轮／空闲 Promise 身份也不同。v4 将原生通知关联真实渲染完成标记，等待队列及原生渲染；post 按批处理，把新增 post 留到下一轮 pre／渲染后；同轮共享完成 Promise、空闲共享已完成 Promise，不凭空增加渲染轮。

最终浏览器前后对照 v3 3/8 一致、v4 8/8 一致；保留旧失败。直接在 nextTick 返回时取 DOM，不在判断前额外等待 San 来掩盖差异。另两张实际控件页输入／勾选／下拉选择及每页 7 条 watcher 回调一致；目标 Vue=false、VueReactivity=true，双方无 warning/error。卸载验证为 owner watcher 取消、既有 tick 回调仍执行，不等于完整生命周期实现。

非 DOM 原生重放 94 项一致、写入 5 项及控制 8 项保持。Node 的 render-null 夹具新调用 San 调度后可打印结果但进程不退出，本地 MessageChannel 分支未关闭 channel；两次运行被中断，不能算完成通过。v4 非 DOM 夹具显式注入微任务渲染屏障替代并记录 node_render_barrier_is_test_double=true；实际 DOM／San 调度结论仅由真实浏览器证明，不修改 engine 或全局 MessageChannel。

最终 v1/v2/v3/v4 合并工程回归 22 项通过（50.132 秒，含旧 17 项和新增 5 项）；既有生产公开键／控件 11 项通过（28.071 秒）。中间专项 5 项 16.708／27.132 秒不累计。未全量重跑系统／30 组件／真实模型，以上不是论文新增迁移样本数。

最终前后浏览器、控件及非 DOM 5 份报告和源码／测试 6 份，共 11 payload，以 SHA256SUMS.json 封存至 migration_system/docs/verification/2026-11-17_public_instance_proxy_v4；详见 migration_system/docs/PUBLIC_INSTANCE_PROXY_V4.md。只有最终包绑定最终源码，中间 JSON 不冒充当前证明。

完整私有 vnode／实例图、父子／循环作用域、生命周期 hooks／异常／调度交错／递归保护和真实 SSM 编译／导出／加载尚未闭合；v3 原生重入及事件口径差异、development 内核触发钩子限制仍保留。不缩小任意公开键契约，arbitrary_public_key_contract_complete=false、production_admission_eligible=false，生产 source_dynamic_key_contract_unsupported 保持。无模型、LangSmith、训练、实际数据库／队列写入、commit 或 push，无需重启或付费复跑。


## 2026-11-17：公开实例代理 v5 root owner 生命周期与清理（研究，未接入）

继续先兼容验证再接入的授权，以 v4 为基线新增 migration_system/research/public_instance_v5。v4 未连接 owner hooks，直接原生 San dispose 会绕过句柄清理；本轮连接同一代理／原生实例的 beforeMount/mounted、beforeUpdate/updated、beforeUnmount/unmounted。hook 的 this 为业务 Proxy，原生 receiver 独立；hook 中 $watch 归 owner scope，原生及句柄销毁走同一清理。旧版本、生产引擎／Prompt／规则／阈值／配置、源 Vue、历史产物和费用不改。

新增有限 root owner 浏览器对照旧 v4 0/6、新 v5 6/6 一致，比较 hook 顺序、this、实际 DOM／$el、状态及 mounted/unmounted／scope 标志。旧 0/6 是新增场景此前未连接，不是将旧测试通过率或论文迁移率改为 0。beforeUnmount 在旧 DOM／活 scope 执行；停止 scope／cleanup 并取消队列后移除 DOM，unmounted 时 scope 已停但 isUnmounted 仍 false，hook 后标 true。保留卸载后的原 $el 引用，不伪造 vnode；mounted 中 watcher 销毁后不再收到变更。

owner 已连接时，原有 8 项实际调度及双向控件、每页 7 条 watcher 回调保持；10 项 owner 边界控制通过，包括父级／transition 字段／真实 s-transition、描述符、重复绑定、外部 detach、借用 receiver、幂等和不可扩展实例。非 DOM 原生 94 项、写入 5 项及 8 项控制保持，继续显式使用渲染屏障测试替代；实际 DOM／调度证据来自隔离 Edge。目标仍显式依赖独立 VueReactivity，但无 Vue 组件运行时。

最终 v1 至 v5 合并工程回归 28 项通过（70.162 秒，含旧 22 项及新增 6 项），生产公开键／控件专项 11 项通过（25.828 秒）。专项 6 项 19.309 秒、补指令防护前合并 28 项 65.115 秒是中间结果，不累计。未全系统／30 组件／真实模型重跑，不作为论文新增迁移样本或完整视觉评估。

保留实际失败：San dispose 内部也调用 detach，首次被外部 detach 防护误挡；仅在 unmounting 状态允许内部路径，外部仍拒绝。封存前补查真实 s-transition 指令反例，再跑合并回归及最终浏览器结果。每实例 dispose／attach 幂等防护保留，不声称所有 wrapper 卸载后均恢复。

七份最终报告与九份源码／测试共 16 payload，以 SHA256SUMS.json 封存至 migration_system/docs/verification/2026-11-17_public_instance_proxy_v5，详见 migration_system/docs/PUBLIC_INSTANCE_PROXY_V5.md。旧 v1 至 v4 封存及生产基线保持；中间文件不能冒充最终源码证据。

完整父子／循环作用域、初始化 hooks／组合式注入、过渡／异步卸载／祖先异常、所有调度交错／递归保护、私有实例图和真实 SSM 自动编译／导出／加载仍未闭合。保持 arbitrary_public_key_contract_complete=false、production_admission_eligible=false 与生产 source_dynamic_key_contract_unsupported；root 夹具只是证据范围，不缩小用户任意公开键契约来放行。无模型、LangSmith、训练、实际数据库／队列写入、commit 或 push，无需重启或付费复跑。


## 2026-11-17：代理原型 v6 父子／循环 owner（3/4，keyed 未解决）

沿用户授权先验证后接入，新增 research/public_instance_v6，不改 v1 至 v5、原封存、生产引擎／Prompt／规则／阈值／配置、源 Vue 和产物。子 owner 根据真实 San runtime.owner 关联已登记父 owner，私有 capability 连接实际逻辑 parent/root，公开 Proxy 身份和独立 scope 对照；family nextTick 等待成员队列。初始使用真实 DocumentFragment 构建后插入，批量销毁按 DOM 顺序清理且保留子 DOM 到父移除，unmounted 子先父后，条件移除排到父 post 队列。

四个有限树场景旧 v5 0/4、新 v6 3/4：父子实际 hook／DOM／身份、父销毁取消子排队 watcher、独立 root 隔离已一致；keyed 重排／删除／重插仍不一致。旧 0/4 是新增树场景缺口，不改写旧测试或论文迁移率。初轮只有 2/4，修正挂载可见性／兄弟卸载顺序后为 3/4，不将部分结果说成全部通过。

keyed 失败为 a/b/c 的局部 count 均为 1，改 c/b 后 San 销毁并重建应保留的 b，count 变 0。另运行不含 Proxy、VueReactivity 或 owner helper 的原生 San 3.15.6 对照，同样 bSame=false、cSame=true、bDisposed=true、DOM 为 c:1/b:0。证据定位此具体失败到原生更新路径，尚未定位内部算法行，也未证明任意 key／模板均失败；不靠忽略 key、仅比 DOM、回填状态或改 Prompt 判通过，不改固定 engine。

既有 root 6 项、调度 8 项、控件每页 7 条实际 watcher 回归保持。非 DOM 原生 94 项、写入 5 项／8 控制和 root 10 控制保持，渲染屏障替代显式声明，实际 DOM 判断来自隔离 Edge。目标仍加载独立 VueReactivity，不加载 Vue 组件运行时。

最终合并工程检查 34 项通过（93.577 秒，含旧 28 项及新增 6 项），其中包含确认 keyed 失败仍被识别的反例断言，因此不等于树行为 4/4；生产专项 11 项通过（30.039 秒）。专项 6 项 20.819 秒是中间结果不累计。未全系统／30 组件／模型重跑，不作为论文迁移样本或完整视觉评估。

最终八份报告和十份源码／测试共 18 payload，以 SHA256SUMS.json 封存至 migration_system/docs/verification/2026-11-17_public_instance_proxy_v6；详见 migration_system/docs/PUBLIC_INSTANCE_PROXY_V6.md。旧中间结果不冒充最终源码证明，旧封存和生产基线保持。

优先下一步定位修复已冻结的原生 keyed 差异，再谈生产整合。动态子 props、任意深度／循环／key、跨独立 root 全局调度、完整私有实例图、初始化 hooks／组合式注入／祖先异常／过渡等及真实 SSM 编译／导出／加载仍未完成；dynamic_child_prop_updates_verified=false。保留 arbitrary_public_key_contract_complete=false、production_admission_eligible=false 和生产 source_dynamic_key_contract_unsupported，不缩小用户契约。无模型、LangSmith、训练、实际数据库／队列写入、commit 或 push，无需重启或付费复跑。


## 2026-11-17：代理研究 v7 修复已封存的 keyed 组件重建（未接入生产）

继续用户允许内部公开实例代理、先验证后接入的授权。本轮在 research/public_instance_v7 中定位并修复 v6 已封存反例；不修改 v1 至 v6／封存、生产 engine、生成／修复 Prompt、规则、阈值、配置、源 Vue 或历史产物。

根因是 San 3.15.6 ForNode._updateArray 只把 ELEM 视为可移动节点；组件不在 LIS 静止项时销毁／重建，a/b/c 改 c/b 使 b 的局部 count 丢失。新 san_candidate.cjs 按精确版本及上游 SHA-256 生成内存候选，在原移动／销毁前检查所有旧组件，只对同步已挂载、同父 DOM、单元素根且无过渡的组件启用原移动路径；不改 node_modules，不按组件名特判，不回填状态。

同一实际树场景 v6 3/4、v7 4/4，无 warning/error。无代理原生最小对照由 bSame=false、bDisposed=true、c:1/b:0，变为 bSame=true、bDisposed=false、c:1/b:1。旧失败报告及源码原样保留，不改写原 v6 记录。

另运行 Vue／上游 San／候选 San 三页原生探针：13 次连续列表更新，每次重建同 key 的对象，验证身份、局部 count、原生 props／index 传播及 DOM 点击；普通元素路径也回归，两项限定案例一致。目标不含 Proxy、VueReactivity 或 owner helper。9 项实际拒绝控制包含 fragment／条件／组件根、指令及属性过渡、重复／保留名／数字／Symbol key；错误时 DOM／旧实例未变，但不承诺撤销已经写入的 Data。循环 key 研究边界不替代用户要求的 setValue 任意公开键契约。

实际中间失败：首轮扩展原生探针整体不一致，逐步字段已一致，仅最终 raw San 整树销毁顺序源 a/b、原 San 与候选 b/a。最终报告分开 keyed 更新与最终销毁观察，保留 native_disposal_order.equal=false，不声称 keyed 补丁修复了整树 dispose；owner 桥接后的真实卸载顺序仍由四项树探针验证。首轮 v7 专项 7 项中 1 项失败、43.116 秒不累计为最终通过。

同一候选下既有 root 6 项、调度 8 项和控件／每页 7 条 watcher 保持。非 DOM 94 项、原生写入 5 项／8 控制、owner 10 控制保持；这些仍使用原 San 和明确的渲染屏障测试替代，不冒充候选 DOM 时序证明。代理页仍使用独立 VueReactivity，没有 Vue 组件运行时，不能宣称完全无 Vue 依赖。

最终 v1 至 v7 合并工程检查 41 项通过（110.122 秒，旧 34 项＋新增 7 项）；生产公开键／控件专项 11 项通过（27.472 秒）。工程通过含已知差异／拒绝边界断言，不等于任意契约通过。九份最终报告、十一份 cjs、一份 Python 共 21 payload 封存到 migration_system/docs/verification/2026-11-17_public_instance_proxy_v7，SHA256SUMS.json 绑定。旧封存及 185 项生产实现基线保持。

代理 child props 动态传播、完整私有实例图、初始化 hooks／组合式注入／祖先异常／过渡、所有调度交错／递归保护及真实 SSM 编译／导出／加载仍未闭合；dynamic_child_prop_updates_verified=false。下一步优先补动态代理子 props 的真实父子同步反例与闭环。arbitrary_public_key_contract_complete=false、production_admission_eligible=false 和生产 source_dynamic_key_contract_unsupported 不变。无真实模型、LangSmith、训练、实际数据库／队列写入、commit/push；无需重启或付费复跑。不作为论文新增迁移样本或完整像素／SSIM 评估。详见 migration_system/docs/PUBLIC_INSTANCE_PROXY_V7.md。


## 2026-11-17：用户确认保持 San 目标，另立业务实例契约

用户明确选择保持 San 迁移目标，不要求目标组件复制 Vue 私有 $／_／vnode／renderer 图。新契约标识 san-business-instance-v1，声明见 migration_system/docs/SAN_BUSINESS_INSTANCE_CONTRACT_V1.json 及同名 Markdown。任意字符串／Symbol 业务键、原生 ToPropertyKey、字面点号键、data/watch 等命名空间冲突和对象／ref／别名行为仍需保留；不缩为源码六键或简单值。公开 API、slots／refs／父子／错误处理仍需实现或明确阻断，不能改称私有差异。依赖私有图或未知动态读取的源码不得静默放行。

这是新的验收范围，不是旧契约修好。v1–v7 源码、封存、失败、旧完整状态、实验分母和费用均保留，不将 94 项旧有限测试自动改写为新契约完成。生产 source_dynamic_key_contract_unsupported 保持；独立响应式内核可以复用但如实披露，目标不挂载 Vue 源组件、不引入完整 Vue 组件／渲染器，也不要求用户额外提供互操作元数据。


## 2026-11-17：业务接收器、自动源编译及真实冻结样本交互重放（部分实现）

新增 migration_system/runtime/san_business_v1：San constructor 中执行源 Options 方法，隔离业务 Proxy 和原生句柄；自动从源 SFC AST 生成业务脚本、refs、属性策略及实现指纹，不在 Node 执行源代码。复用已验证的响应式／owner 思路，补简单动态父 props、watch／computed／hooks DOM 顺序、$emit 的配置回调／once／model 策略、forceUpdate 去重、共享内核及卸载后 scope。跨独立导出共享内核和 owner 品牌，业务同名 Symbol 不覆盖原生登记。目标依赖固定 San 3.15.6，内嵌独立 @vue/reactivity／@vue/shared 3.5.45（含许可头），后者用于 toDisplayString；不是零 Vue 依赖，但不加载 Vue 组件／渲染器。

实际反例及修复：首轮冻结样本 value／空 class DOM 属性缺失且 key 表达不一致，随后在真实 DOM 更新中落实属性，并仅在编译器独立核实源 :key／目标 trackBy 完全一致后，版本化投影模板身份；不改原始 DOM 或阈值。Windows 文本写入曾改变换行、破坏导出字节 SHA，改为精确 UTF-8 并加入回归。卸载后 watcher 早期错误地继续回调，固定版本 Vue 对照表明允许注册但生成不活跃 watcher，按此修复。新增 attrs 反例还发现 display-name 未归一化、未声明 onClick 被误删及 key 被错误计入 attrs；修复前观察保留，现已按声明 emits 分类。未实现的根 ref／vnode 回调及原生父事件路由明确拒绝，不静默丢弃。

本轮专项 18 项通过（37.111 秒）；旧 v1–v7 工程回归 41 项通过（76.907 秒）；默认生产公开键边界专项 11 项通过（19.396 秒）。这些是不同层级的工程断言，不累加为迁移率。新原生对照包含 92 个有限业务案例，2 个私有观察排除项分开；源自动编译控件一组、5 个公共 API／父子夹具、12 类有限值显示和 4 个共享／拒绝控制分别记录，不冒充任意输入等价或真实组件树自动编译。

同一冻结 ProductFilterPanel source.vue 与原第 3 版 San 模板，经离线编译派生新产物；搜索、分类、库存、排序、价格、选中、清空及空结果共 11 步真实隔离浏览器动作，初始和每步 DOM／样式／控件状态检查通过，初始结构／原始 DOM 相似度 1，UI 100%，SSIM 1，不同像素 0。静态范围仅为规范源 Options／接收者脚本绑定，不冒称旧完整语义验证器已适配；旧失败和产物不回写，原 0.95／0.95／0.05 阈值保持。

复现入口 migration_system/scripts/verify_san_business.py；源码副本、输入、18／41／11 日志、探针、11 步重放和 SHA256SUMS 封存至 migration_system/docs/verification/2026-11-17_san_business_instance_v1_partial，具体以 summary.json 和日志为准。此前中间重放报告不冒充最终源码绑定；部分早期导出有已记录的 Windows 换行问题。详见 migration_system/docs/SAN_BUSINESS_RUNTIME_V1.md。

完整任务尚未完成：默认生成／校验／修复、候选保护、任务指纹、API／前端未接入；自动 source／SSM 组件树、父原生事件参数／祖先错误捕获、完整动态 slots／attrs／refs、递归 watcher 保护及带副作用 getter／coercion 顺序仍待闭合。新代码不能冒充完整任意键契约发布，full_contract_complete=false、production_admission_eligible=false。默认 185 文件生产基线、engine 原件、旧 v1–v7 封存和用户配置保持；无真实模型、LangSmith、训练、实际业务库／队列写入、commit/push，目前无需重启或付费复跑。


## 2026-11-17：自动内联源树、事件／错误与实际读取时序（仍未完成完整契约）

在首轮18项、47 payload封存后继续，不覆盖旧证据。新增 template.cjs／tree.cjs／events.cjs：静态const内联Options、template和注册表从源AST生成San树，原方法在业务Proxy上执行，不在Node执行源代码，不加载Vue组件／渲染器。真实SFC编译对照覆盖两参数／数组／once、动态props／inheritAttrs=false的attrs及原连字符名、组件refs／parent／root、子$options与父组件注册对象身份、祖先errorCaptured返回false。不是手工父子metadata成功的重命名。

修正新反例：严格this身份比较误报逃逸；data-note被San归一化而破坏$attrs；过早镜像读getter；San局部更新漏掉Vue整组件渲染读；未用字段误触发渲染／旧分支依赖未清理；任意键覆盖方法后事件立即换绑而非渲染时切换；可选成员调用错误绑定严格函数this；回调数组被动态替换后没有渲染依赖。实现惰性失效、实际渲染读记录及清理、渲染前非循环成员快照与响应式失效、实际DOM节点事件快照、统一事件／watch／hook错误路由。

值探针为12类值加一个副作用toString；公共API夹具6项，新增单根pre watcher递归上限的原生对照：101次回调、count=102、nextTick拒绝并不提交待渲染DOM。源自动控件探针包含getter、未用字段及隐藏分支。工程结果不等于任意值／全局调度／所有模板顺序已等价。

最终同一实现指纹下专项20项通过（32.673秒）、旧v1–v7回归41项通过（63.472秒）、默认生产公开键边界11项通过（15.044秒）；六份探针、源代码副本和冻结商品筛选11步交互结果封存至 migration_system/docs/verification/2026-11-17_san_business_instance_v1_tree_partial。商品筛选结构／原始DOM／样式／控件检查通过，UI100%、SSIM1、不同像素0，阈值不变；不回算旧迁移率或费用。详见 migration_system/docs/SAN_BUSINESS_RUNTIME_V1.md。

本轮仍未完成完整契约：slots、动态／函数／循环refs、自动attrs透传、外部／递归组件和更宽模板、跨root／复杂异步递归及循环／条件事件getter和DOM先后仍未闭合；主链生成／验证／修复／候选门及独立发布绑定未接入。尤其research v7用加载前San内存候选通过的keyed4/4，不能自动算为本导出通过，本导出尚使用原San，stateful keyed重排需移植并独立复测。full_contract_complete=false、production_admission_eligible=false和默认source_dynamic_key_contract_unsupported保持。原23格不改，只追加此格；旧源／185文件生产基线／封存不改，无模型／LangSmith／训练／实际业务库或队列写入／commit/push，不要求现在重启或付费复跑。
